In [ ]:
# ============================
# 1. Install dependencies
# ============================
# !pip install pymupdf faiss-cpu sentence-transformers gradio matplotlib seaborn openai

# ===============================================================
# RAG Chatbot over instructors CSV
# Auto image = the person your ANSWER is about (retrieval + answer check)
# ===============================================================

import os, re, io, time
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass

import numpy as np
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import gradio as gr
from openai import OpenAI

# Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# -------------------------------
# Configuration
# -------------------------------

# Update these paths to your actual files
CSV_FILE = "data/instructors.csv"  # Path to your CSV file

INSTRUCTOR_IMAGES = {
    "instructor1": "images/instructor1.jpg",
    "instructor2": "images/instructor2.jpg",
    "instructor3": "images/instructor3.jpg",
}

# Aliases used to detect identity in queries, context, and answers
INSTRUCTOR_ALIASES = {
    "instructor1":   [r"\binstructor1\b", r"first\s+instructor\b", r"instructor\s+one"],
    "instructor2": [r"\binstructor2\b", r"second\s+instructor\b"],
    "instructor3": [r"\binstructor3\b", r"third\s+instructor\b"],
}

# Map exact CSV "name" to our keys
NAME_TO_KEY = {
    "First Instructor Name": "instructor1",
    "Second Instructor Name": "instructor2",
    "Third Instructor Name": "instructor3",
}

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
RETRIEVE_K = 3
OPENAI_MODEL = "gpt-4o-mini"
MAX_TOKENS = 350

# -------------------------------
# Helpers
# -------------------------------

def require_api_key():
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key or key == "your-openai-api-key-here":
        raise RuntimeError("Please set OPENAI_API_KEY in your environment or update the code above.")
    return key

@dataclass
class SourceChunk:
    text: str
    source: str  # row name, e.g., "First Instructor Name"

# This is for FAISS normalization
def normalize_rows(v: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v / norms

def detect_identity_by_alias(text: str) -> Optional[str]:
    tl = text.lower()
    for key, patterns in INSTRUCTOR_ALIASES.items():
        for pat in patterns:
            if re.search(pat, tl):
                return key
    return None

# -------------------------------
# Build RAG Index from CSV
# -------------------------------

class RAGIndex:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.chunks: List[SourceChunk] = []
        self.embs: Optional[np.ndarray] = None

    def build(self, csv_file: str):
        df = pd.read_csv(csv_file)
        for _, row in df.iterrows():
            text = f"""
            Name: {row['name']}
            Title: {row['title']}
            Company: {row['company']}
            Skills: {row['skills']}
            Bio: {row['bio']}
            Fun Facts: {row['fun_things']}
            LinkedIn: {row['linkedin_url']}
            """
            self.chunks.append(SourceChunk(text=text.strip(), source=str(row['name']).strip()))

        texts = [c.text for c in self.chunks]
        embs = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
        embs = normalize_rows(embs)
        self.embs = embs

        dim = embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embs)

    def search(self, query: str, k: int = RETRIEVE_K) -> List[Tuple[SourceChunk, float]]:
        q = self.model.encode([query], convert_to_numpy=True)
        q = normalize_rows(q)
        scores, idxs = self.index.search(q, k)
        return [(self.chunks[i], float(s)) for i, s in zip(idxs[0], scores[0]) if i != -1]

# -------------------------------
# Chatbot
# -------------------------------

class InstructorCSVChatbot:
    def __init__(self, csv_file, images):
        require_api_key()
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.rag = RAGIndex(EMBEDDING_MODEL_NAME)
        self.rag.build(csv_file)
        self.images = images
        self.history = []

    # --- Person resolution used for image selection ---
    def _infer_person_from_hits(self, hits: List[Tuple[SourceChunk, float]]) -> Optional[str]:
        """Score people by retrieval similarity; prefer source name mapping, then alias in text."""
        scores_by_person: Dict[str, float] = {}
        for chunk, sc in hits:
            # 1) try source name (CSV row)
            key = NAME_TO_KEY.get(chunk.source)
            # 2) fallback: alias within the chunk text
            if not key:
                key = detect_identity_by_alias(chunk.text)
            if key:
                scores_by_person[key] = scores_by_person.get(key, 0.0) + max(sc, 0.0)
        if not scores_by_person:
            return None
        # pick the person with highest accumulated score
        return max(scores_by_person.items(), key=lambda kv: kv[1])[0]

    def _infer_person_from_answer(self, answer: str) -> Optional[str]:
        """If the model explicitly mentions a person, use that."""
        return detect_identity_by_alias(answer or "")

    def _choose_image_for_answer(self, hits, answer: str) -> Optional[str]:
        """Final decision: use answer mention if present; else use retrieval-majority person."""
        person_from_answer = self._infer_person_from_answer(answer)
        if person_from_answer and person_from_answer in self.images and os.path.exists(self.images[person_from_answer]):
            return self.images[person_from_answer]

        person_from_hits = self._infer_person_from_hits(hits)
        if person_from_hits and person_from_hits in self.images and os.path.exists(self.images[person_from_hits]):
            return self.images[person_from_hits]

        return None

    # --- LLM answering ---
    def _answer(self, question: str, context: str) -> str:
        system = "You are a helpful assistant that must answer ONLY from the instructor CSV context. If it's not in the context, say you don't know."
        user = f"""Context:
{context}

Question: {question}
"""
        resp = self.client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0.2,
            max_tokens=MAX_TOKENS,
        )
        return resp.choices[0].message.content.strip()

    def chat(self, question: str):
        hits = self.rag.search(question)
        ctx = "\n\n".join([f"{c.source}: {c.text}" for c, _ in hits])
        answer = self._answer(question, ctx)

        # Decide image based on WHO the answer is about
        img_path = self._choose_image_for_answer(hits, answer)

        self.history.append((question, answer))
        return answer, img_path

# -------------------------------
# Gradio UI
# -------------------------------

bot = InstructorCSVChatbot(CSV_FILE, INSTRUCTOR_IMAGES)

def on_chat(msg, history, img_state):
    ans, img_path = bot.chat(msg)
    history = history + [(msg, ans)]
    img_state = img_path if img_path else None
    return "", history, img_state

with gr.Blocks() as demo:
    gr.Markdown("# Instructor Chatbot")
    with gr.Row():
        chatbot_ui = gr.Chatbot(label="Chat with our instructors", height=420)
        img = gr.Image(label="Instructor Photo", interactive=False)
    with gr.Row():
        txt = gr.Textbox(show_label=False, placeholder="Ask about our instructors...")
        send = gr.Button("Send")

    send.click(on_chat, [txt, chatbot_ui, img], [txt, chatbot_ui, img])

if __name__ == "__main__":
    require_api_key()
    demo.launch(share=True)